In [1]:
from datasets import load_dataset, Dataset, DatasetDict
from collections import Counter
from tqdm import tqdm
from create_ilm_examples import randomly_mask_document
import ilm.mask
from ilm.mask.util import mask_cls_str_to_type
import multiprocessing as mp
import nltk 

/home/bkaraca/Desktop/table2text/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to /home/bkaraca/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/bkaraca/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
ds = load_dataset("michaelauli/wiki_bio", trust_remote_code=True)

In [4]:
ds.column_names

{'train': ['input_text', 'target_text'],
 'test': ['input_text', 'target_text'],
 'val': ['input_text', 'target_text']}

In [5]:
len(ds["train"])

582659

In [6]:
_WORKER_MASKER = None
_WORKER_NUM_EXAMPLES_PER_DOCUMENT = None
_WORKER_MAX_NUM_RETRIES = None
_WORKER_KWARGS = None


def _init_mask_worker(mask_cls, mask_arg0, num_examples_per_document, max_num_retries, kwargs):
  global _WORKER_MASKER
  global _WORKER_NUM_EXAMPLES_PER_DOCUMENT
  global _WORKER_MAX_NUM_RETRIES
  global _WORKER_KWARGS

  mask_type = mask_cls_str_to_type(mask_cls)
  if mask_arg0 is None:
    _WORKER_MASKER = mask_type()
  else:
    _WORKER_MASKER = mask_type(mask_arg0)

  _WORKER_NUM_EXAMPLES_PER_DOCUMENT = num_examples_per_document
  _WORKER_MAX_NUM_RETRIES = max_num_retries
  _WORKER_KWARGS = kwargs


def _normalize_doc_masks(doc_masks):
  normalized = []
  for example in doc_masks:
    normalized_example = []
    for span_type, offset, length in example:
      normalized_example.append({
          "type": span_type.name.lower(),
          "offset": int(offset),
          "length": int(length),
      })
    normalized.append(normalized_example)
  return normalized


def _mask_one_document(doc):
  doc_masks, error_to_count = randomly_mask_document(
      doc,
      _WORKER_MASKER,
      _WORKER_NUM_EXAMPLES_PER_DOCUMENT,
      _WORKER_MAX_NUM_RETRIES,
      **_WORKER_KWARGS)
  return _normalize_doc_masks(doc_masks), error_to_count


def randomly_mask_dataset(
    ds: Dataset,
    mask_cls='ilm.mask.hierarchical.MaskHierarchical',
    mask_arg0=None,
    num_examples_per_document=16,
    max_num_retries=16,
    num_workers=mp.cpu_count(),
    chunksize=32,
    tqdm=tqdm,
    **kwargs):
  docs = ds["target_text"][:]
  input_texts = ds["input_text"][:]

  all_doc_masks = []
  error_to_count_total = Counter()

  if not num_workers or num_workers <= 1:
    mask_type = mask_cls_str_to_type(mask_cls)
    if mask_arg0 is None:
      masker = mask_type()
    else:
      masker = mask_type(mask_arg0)

    for doc in tqdm(docs):
      doc_masks, error_to_count = randomly_mask_document(
          doc,
          masker,
          num_examples_per_document,
          max_num_retries,
          **kwargs)
      all_doc_masks.append(_normalize_doc_masks(doc_masks))
      for k, v in error_to_count.items():
        error_to_count_total[k] += v
  else:
    with mp.Pool(
        processes=num_workers,
        initializer=_init_mask_worker,
        initargs=(
            mask_cls,
            mask_arg0,
            num_examples_per_document,
            max_num_retries,
            kwargs,
        )) as pool:
      for doc_masks, error_to_count in tqdm(
          pool.imap(_mask_one_document, docs, chunksize=chunksize),
          total=len(docs)):
        all_doc_masks.append(doc_masks)
        for k, v in error_to_count.items():
          error_to_count_total[k] += v

  out_ds = Dataset.from_dict({
      "input_text": input_texts,
      "target_text": docs,
      "masked_spans": all_doc_masks,
  })
  return out_ds, error_to_count_total

In [7]:
ds_masked1 = randomly_mask_dataset(ds["test"], num_workers=1)

100%|██████████| 72831/72831 [14:57<00:00, 81.18it/s] 


ArrowTypeError: Expected bytes, got a 'list' object

In [ ]:
ds_masked2 = randomly_mask_dataset(ds["test"], num_workers=8)

In [8]:
ds_masked3 = randomly_mask_dataset(ds["test"], num_workers=mp.cpu_count())

100%|██████████| 72831/72831 [03:42<00:00, 327.44it/s]


ArrowTypeError: Expected bytes, got a 'list' object